# FIRMS fire alerts and fire count grid, Kalimantan, Sumatera and the Legal Amazon, August to October 2026

Two areas, each in its own subplot of every figure: Kalimantan with mainland Sumatera, and the 9 states of the Brazilian Legal Amazon. For each area the notebook plots the provinces or states on a basemap, the FIRMS fire alerts made by `job/firms` (VIIRS and MODIS near real time, clipped to each province or state) with the 2020 forest and the large cities, the fire count grids made by `job/firms_grid` (number of alerts per cell, with sea, outside the area and WorldCover water masked), and the number of alerts per regency or municipality (admin level 2).

The provinces and states come from `forest_fire.aoi`, the same definitions as the jobs. Brazil boundaries are GADM 4.1, free for non-commercial use only (https://gadm.org/license.html).

In [ ]:
import os

os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebook" else os.getcwd())

import re
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from dotenv import load_dotenv
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from rasterio.enums import Resampling

from forest_fire.aoi import REGIONS, gdal_path, read_provinces

# override=True so the .env credentials win over anything already in the environment
load_dotenv(override=True)

# Lower resolution figures, keeps the notebook small
plt.rcParams["figure.dpi"] = 60
# Base text size, every other size in the plots is relative to it ("small", "large", ...)
plt.rcParams["font.size"] = 18

# Areas shown side by side in every figure, each made of regions of forest_fire.aoi
AREAS = {
    "Kalimantan and Sumatera": ["sumatera", "kalimantan"],
    "Legal Amazon": ["legal_amazon"],
}
# Admin level 2 of each area, with the fields for the id, the name and the province or state
ADM2 = {
    "Kalimantan and Sumatera": dict(
        url="https://storage.googleapis.com/gee-ramiqcom-bucket/admin/indonesia_adm_level_2.fgb",
        id="KDPKAB",
        name="WADMKK",
        province="WADMPR",
        unit="regency or city",
    ),
    # GADM 4.1 municipalities of Brazil, free for non-commercial use only
    "Legal Amazon": dict(
        url="https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BRA_2.json.zip",
        id="GID_2",
        name="NAME_2",
        province="NAME_1",
        unit="municipality",
    ),
}
CLOUD_PREFIX = "gs://gee-ramiqcom-s4g-bucket/forest_fire_indonesian_2026"
DATE_NAME = "2026-08_2026-10"
GFC_PREFIX = f"{CLOUD_PREFIX}/jrc_gfc_2020"
PLACES_URL = "/vsicurl/https://storage.googleapis.com/gee-ramiqcom-s4g-bucket/forest_fire_indonesian_2026/urban/ne_10m_populated_places.fgb"
# Populated places shown on the maps
MIN_POPULATION = 1_000_000
# Width of one subplot in inches, the height follows the shape of the area. Narrower maps keep
# the text readable when the figure is shrunk to fit a page or a post
MAP_WIDTH = 14
# Figures saved for reports
FIGURE_DIR = Path("output/figures")

## Map helpers

In [ ]:
def add_lonlat(ax):
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{abs(x):.2f}°{'E' if x >= 0 else 'W'}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{abs(y):.2f}°{'N' if y >= 0 else 'S'}"))
    ax.tick_params(labelsize="small")
    # The ticks already say °E and °N, axis titles only take space between the maps
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(color="grey", linewidth=0.4, alpha=0.6, linestyle="--")


def add_north_arrow(ax, x=0.94, y=0.92):
    ax.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - 0.1),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize="large",
        fontweight="bold",
        arrowprops=dict(facecolor="black", edgecolor="white", width=4, headwidth=12),
    )


def add_scalebar(ax, x=0.05, y=0.05):
    # Axes are in degrees, so convert km to degrees of longitude at the map's center latitude
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    km_per_deg = 111.32 * np.cos(np.radians((ymin + ymax) / 2))
    width_km = (xmax - xmin) * km_per_deg
    length_km = next(n for n in (500, 200, 100, 50, 20, 10, 5, 2, 1, 0.5) if n <= width_km / 4)
    length_deg = length_km / km_per_deg

    x0 = xmin + x * (xmax - xmin)
    y0 = ymin + y * (ymax - ymin)
    height = (ymax - ymin) * 0.012
    for i in range(2):
        ax.add_patch(
            plt.Rectangle(
                (x0 + i * length_deg / 2, y0),
                length_deg / 2,
                height,
                facecolor="black" if i == 0 else "white",
                edgecolor="black",
                linewidth=0.8,
            )
        )
    for frac in (0, 0.5, 1):
        ax.text(
            x0 + frac * length_deg,
            y0 + height * 1.6,
            f"{length_km * frac:g}",
            ha="center",
            va="bottom",
            fontsize="small",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1),
        )
    ax.text(x0 + length_deg + (xmax - xmin) * 0.01, y0, "km", va="bottom", fontsize="small")


def padded_bounds(gdf, pad_frac=0.03):
    xmin, ymin, xmax, ymax = gdf.total_bounds
    pad = pad_frac * max(xmax - xmin, ymax - ymin)
    return xmin - pad, ymin - pad, xmax + pad, ymax + pad


def set_extent(ax, gdf):
    xmin, ymin, xmax, ymax = padded_bounds(gdf)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    # Degrees drawn equal, the axes box from area_figure has the same shape so it is filled
    ax.set_aspect("equal")


def add_basemap(ax, zoom_offset=1, **kwargs):
    # Light hillshade, one zoom level more detailed than contextily picks for the map extent (axes in degrees)
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    zoom = cx.tile._calculate_zoom(xmin, ymin, xmax, ymax) + zoom_offset
    cx.add_basemap(
        ax, crs="EPSG:4326", source=cx.providers.Esri.WorldShadedRelief, zoom=zoom, attribution_size=10, **kwargs
    )


def decorate(ax, title):
    add_lonlat(ax)
    add_north_arrow(ax)
    add_scalebar(ax)
    ax.set_title(title)


def area_figure(ncols=1, map_width=MAP_WIDTH, colorbar=None):
    """One row of maps per area, placed exactly so the gap between areas stays small.

    Each map box has the shape of its area, so nothing is left empty around it. ``colorbar``
    adds color bar axes: ``"shared"`` one for the figure, ``"each"`` one next to each map.
    Returns the figure, the maps per area and the color bar axes (one, per area or None).
    """
    # Space in inches for titles, tick labels and color bars, larger with the text size
    scale = plt.rcParams["font.size"] / 10
    left, right, top, bottom, row_gap, col_gap = (v * scale for v in (0.75, 0.3, 0.45, 0.3, 0.1, 1.0))
    cbar_width, cbar_pad, cbar_label = (v * scale for v in (0.25, 0.2, 0.9))
    cbar_space = cbar_pad + cbar_width + cbar_label
    # Color bars a little away from the maps, city labels near the right edge stay readable
    each = cbar_space + 2 * cbar_pad if colorbar == "each" else 0
    shared = cbar_space + 2 * cbar_pad if colorbar == "shared" else 0

    heights = []
    for area in AREAS:
        xmin, ymin, xmax, ymax = padded_bounds(provinces[area])
        heights.append(map_width * (ymax - ymin) / (xmax - xmin))
    width = left + ncols * (map_width + each) + (ncols - 1) * col_gap + shared + right
    height = sum(top + h + bottom for h in heights) + (len(heights) - 1) * row_gap
    fig = plt.figure(figsize=(width, height))

    def rect(x, y, w, h):
        # Inches from the top left corner to a figure fraction rectangle
        return [x / width, (height - y - h) / height, w / width, h / height]

    axes, caxes = {}, {}
    y = 0
    for area, h in zip(AREAS, heights):
        y += top
        axes[area], caxes[area] = [], []
        for col in range(ncols):
            x = left + col * (map_width + each + col_gap)
            axes[area].append(fig.add_axes(rect(x, y, map_width, h)))
            if colorbar == "each":
                caxes[area].append(fig.add_axes(rect(x + map_width + 3 * cbar_pad, y + 0.15 * h, cbar_width, 0.7 * h)))
        y += h + bottom + row_gap

    if colorbar == "shared":
        x = left + ncols * map_width + (ncols - 1) * col_gap + 3 * cbar_pad
        caxes = fig.add_axes(rect(x, 0.25 * height, cbar_width, 0.5 * height))
    elif colorbar is None:
        caxes = None
    return fig, axes, caxes


def period_title(title):
    return f"{title} {DATE_NAME.replace('_', ' to ')}"

## 1. Provinces and states

Kalimantan and mainland Sumatera provinces from the Indonesian admin level 1 file (Kepulauan Riau and Kepulauan Bangka Belitung not included), and the 9 Legal Amazon states from GADM (Maranhão only partly in the Legal Amazon but taken whole). Each province or state is dissolved into one shape.

In [ ]:
LOWER = "a-zà-ÿ"
UPPER = "A-ZÀ-Þ"


def display_name(name):
    # GADM writes names without spaces, MatoGrosso becomes Mato Grosso, SãoFélixdoXingu São Félix do Xingu
    name = re.sub(rf"(?<=[{LOWER}])(d[aeo]s?)(?=[{UPPER}])", r" \1 ", name)
    return re.sub(rf"(?<=[{LOWER}])(?=[{UPPER}])", " ", name)


provinces = {}
for area, regions in AREAS.items():
    parts = []
    for region in regions:
        part = read_provinces(region)
        part["name"] = part[REGIONS[region]["name_field"]].map(display_name)
        parts.append(part[["name", "geometry"]])
    provinces[area] = gpd.GeoDataFrame(pd.concat(parts), crs="EPSG:4326")

pd.concat({area: gdf.drop(columns="geometry") for area, gdf in provinces.items()})

In [ ]:
fig, axes, _ = area_figure()

for area, (ax,) in axes.items():
    provinces[area].boundary.plot(ax=ax, color="red", linewidth=1.2)
    for _, row in provinces[area].iterrows():
        point = row.geometry.representative_point()
        ax.text(
            point.x,
            point.y,
            row["name"],
            ha="center",
            fontsize="small",
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2),
        )

    set_extent(ax, provinces[area])
    add_basemap(ax)
    decorate(ax, f"{area} provinces and states")

plt.show()

## 2. FIRMS fire alerts, forest and large cities

All alerts from the province and state files. Each point is the centre of a satellite pixel with an active fire detection (375 m for VIIRS, 1 km for MODIS), not a burned area. MODIS near real time has no data for early August, so MODIS points cover only part of the period.

Forest is the JRC Global Forest Cover 2020 v4 made by `job/jrc_gfc_2020` (100 m per province or state, European Commission, CC BY 4.0), read here at about 1 km with the most common class per cell, which is enough at this map scale. Forest in 2020 does not mean forest during the 2026 fires. Cities are the Natural Earth populated places with a population (`POP_MAX`, the metropolitan estimate of the dataset) above 1 million within the map, also outside the area for orientation.

In [ ]:
firms = {
    area: gpd.GeoDataFrame(
        pd.concat(
            [
                gpd.read_file(
                    f"{CLOUD_PREFIX}/firms/{province}_{DATE_NAME}.fgb", columns=["acq_date", "instrument", "satellite"]
                ).assign(province=province)
                for province in provinces[area].index
            ],
            ignore_index=True,
        ),
        crs="EPSG:4326",
    )
    for area in AREAS
}

for area, alerts in firms.items():
    print(f"{area}: {len(alerts):,} alerts, {alerts['acq_date'].min()} to {alerts['acq_date'].max()}")
pd.concat({area: alerts.groupby(["province", "instrument"]).size().unstack(fill_value=0) for area, alerts in firms.items()})

In [ ]:
# Forest per province or state, read at about 1 km (10 times coarser than the 100 m files) to keep memory low
GFC_SCALE = 10

forest_layers = {}
for area in AREAS:
    forest_layers[area] = []
    for province in provinces[area].index:
        with rasterio.open(f"{GFC_PREFIX}/{province}.tif") as src:
            shape = (src.height // GFC_SCALE, src.width // GFC_SCALE)
            # The nodata (255, outside the province) is left out of the most common class
            gfc = src.read(1, out_shape=shape, resampling=Resampling.mode, masked=True)
            bounds = src.bounds
        forest_layers[area].append((gfc.filled(0) == 1, (bounds.left, bounds.right, bounds.bottom, bounds.top)))
    cells = sum(int(forest.sum()) for forest, _ in forest_layers[area])
    print(f"{area}: {len(forest_layers[area])} forest layers, {cells:,} forest cells of about 1 km")


# Natural Earth has a few places sharing the name and population of another, like a Vila Velha in Amapá
# with the population of Vila Velha in Espírito Santo, so the population can't be trusted for either
places_table = gpd.read_file(PLACES_URL, columns=["NAME", "POP_MAX"], ignore_geometry=True)
ambiguous = set(
    places_table[places_table.duplicated(["NAME", "POP_MAX"], keep=False)][["NAME", "POP_MAX"]].itertuples(index=False)
)
print(f"{len(ambiguous)} name and population pairs shared by several places left out")


def read_cities(gdf):
    # Large cities within the map, also outside the area for orientation
    xmin, ymin, xmax, ymax = gdf.total_bounds
    pad = 0.03 * max(xmax - xmin, ymax - ymin)
    places = gpd.read_file(
        PLACES_URL, bbox=(xmin - pad, ymin - pad, xmax + pad, ymax + pad), columns=["NAME", "ADM0NAME", "POP_MAX"]
    )
    places = places[places["POP_MAX"] > MIN_POPULATION].sort_values("POP_MAX", ascending=False)
    places = places[[(name, pop) not in ambiguous for name, pop in zip(places["NAME"], places["POP_MAX"])]]
    # Drop cities within 0.5 degree (about 55 km) of a larger one, Bekasi is part of Greater Jakarta
    kept = []
    for index, city in places.iterrows():
        if all(city.geometry.distance(places.loc[other].geometry) > 0.5 for other in kept):
            kept.append(index)
    return places.loc[kept]


cities = {area: read_cities(provinces[area]) for area in AREAS}
pd.concat({area: gdf.drop(columns="geometry") for area, gdf in cities.items()})

In [ ]:
INSTRUMENT_COLORS = {"VIIRS": "#ffb000", "MODIS": "#ff2a00"}
FOREST_STYLE = dict(face="#2e8b57", edge="#0b4d23", alpha=0.25)
CITY_STYLE = dict(marker="s", size=90, face="white", edge="black")


def add_forest(ax, area, zorder=2):
    # Forest as a low opacity fill with a darker outline along its edge
    for forest, extent in forest_layers[area]:
        ax.imshow(
            np.ma.masked_where(~forest, forest),
            extent=extent,
            cmap=ListedColormap([FOREST_STYLE["face"]]),
            alpha=FOREST_STYLE["alpha"],
            interpolation="nearest",
            zorder=zorder,
        )
        ax.contour(
            forest.astype("uint8"),
            levels=[0.5],
            extent=extent,
            origin="upper",
            colors=FOREST_STYLE["edge"],
            linewidths=0.3,
            zorder=zorder,
        )


def add_cities(ax, area, zorder=6):
    ax.scatter(
        cities[area].geometry.x,
        cities[area].geometry.y,
        s=CITY_STYLE["size"],
        marker=CITY_STYLE["marker"],
        facecolor=CITY_STYLE["face"],
        edgecolor=CITY_STYLE["edge"],
        linewidths=1.2,
        zorder=zorder,
    )
    for _, city in cities[area].iterrows():
        ax.annotate(
            city["NAME"],
            (city.geometry.x, city.geometry.y),
            xytext=(6, 4),
            textcoords="offset points",
            fontsize="small",
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1),
            zorder=zorder,
        )


FOREST_HANDLE = Patch(
    facecolor=FOREST_STYLE["face"] + "40", edgecolor=FOREST_STYLE["edge"], label="Forest 2020 (JRC GFC 2020)"
)
CITY_HANDLE = Line2D(
    [],
    [],
    linestyle="none",
    marker=CITY_STYLE["marker"],
    markersize=10,
    markerfacecolor=CITY_STYLE["face"],
    markeredgecolor=CITY_STYLE["edge"],
    label=f"City above {MIN_POPULATION:,} people (Natural Earth)",
)
LEGEND = dict(loc="upper right", bbox_to_anchor=(0.9, 1), fontsize="small")

fig, axes, _ = area_figure()

for area, (ax,) in axes.items():
    add_forest(ax, area)
    # Scatter on the raw coordinates, much faster than GeoDataFrame.plot for millions of points
    for instrument, color in INSTRUMENT_COLORS.items():
        points = firms[area][firms[area]["instrument"] == instrument]
        ax.scatter(
            points.geometry.x,
            points.geometry.y,
            s=0.3,
            color=color,
            linewidths=0,
            label=f"{instrument} ({len(points):,})",
            rasterized=True,
            zorder=4,
        )
    provinces[area].boundary.plot(ax=ax, color="dimgrey", linewidth=0.8, zorder=5)
    add_cities(ax, area)

    set_extent(ax, provinces[area])
    add_basemap(ax, zorder=1)
    decorate(ax, period_title(f"{area} FIRMS fire alerts, forest and large cities"))

    handles, _ = ax.get_legend_handles_labels()
    legend = ax.legend(handles=handles + [FOREST_HANDLE, CITY_HANDLE], **LEGEND)
    # The alert points are tiny on the map, larger in the legend so their color can be read
    for handle in legend.legend_handles[: len(INSTRUMENT_COLORS)]:
        handle.set_sizes([90])

plt.show()

## 3. Fire count grid

Number of FIRMS alerts per grid cell (0.009°, about 1 km) from `job/firms_grid`, one grid per region, with one color scale for both areas so they can be compared. The grid count only includes alerts within half a cell from the cell centre, so it counts fewer alerts than the points above. Cells with no alerts (0) are shown as no data, as are the masked sea, outside the area and WorldCover water cells. Log color scale because a few cells have far more alerts than the rest. The 2020 forest and the large cities are drawn as in the map above.

In [ ]:
grids = {}
for area, regions in AREAS.items():
    grids[area] = []
    for region in regions:
        with rasterio.open(f"{CLOUD_PREFIX}/firms_grid/{region}_{DATE_NAME}.tif") as src:
            grid = src.read(1)
            bounds = src.bounds
            nodata = src.nodata
            resolution = src.res

        # Masked cells and cells without alerts are not shown
        fire_count = np.ma.masked_where((grid == nodata) | (grid == 0), grid)
        grids[area].append((fire_count, (bounds.left, bounds.right, bounds.bottom, bounds.top)))

        print(
            f"{region}: cell size {resolution[0]:.4f}°, {fire_count.count():,} cells with alerts, "
            f"{int(fire_count.sum()):,} alerts counted, max {fire_count.max()} per cell"
        )

# One color scale for all areas so they can be compared
grid_max = max(fire_count.max() for layers in grids.values() for fire_count, _ in layers)

In [ ]:
# YlOrRd without its near white start, so cells with few alerts stay visible on the light basemap
grid_cmap = ListedColormap(plt.get_cmap("YlOrRd")(np.linspace(0.3, 1, 256)))
grid_norm = LogNorm(vmin=1, vmax=grid_max)

fig, axes, cax = area_figure(colorbar="shared")

for area, (ax,) in axes.items():
    add_forest(ax, area, zorder=2)
    for fire_count, extent in grids[area]:
        image = ax.imshow(fire_count, extent=extent, cmap=grid_cmap, norm=grid_norm, interpolation="nearest", zorder=3)
    provinces[area].boundary.plot(ax=ax, color="dimgrey", linewidth=0.8, zorder=4)
    add_cities(ax, area)

    set_extent(ax, provinces[area])
    add_basemap(ax, zorder=1)
    decorate(ax, period_title(f"{area} FIRMS fire count per cell, forest and large cities"))
    ax.legend(handles=[FOREST_HANDLE, CITY_HANDLE], **LEGEND)

# One color bar, both areas share the color scale
fig.colorbar(image, cax=cax, label="Fire alerts per cell (about 1 km)")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_DIR / f"fire_count_per_cell_{DATE_NAME}.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Fire alerts per regency and municipality

Admin level 2 units: regencies and cities (kabupaten and kota) from the Indonesian admin level 2 file, and municipalities (municípios) from GADM 4.1 for Brazil, with the same province and state filters as the jobs. Each FIRMS alert is assigned to the unit it falls in, so every alert is counted, unlike the grid above. The units differ a lot in size, so the alerts per 100 km² are shown in a second figure after the total count, with the area computed in an equal area projection (EPSG:6933). A few unnamed Indonesian admin level 2 polygons are dropped by the dissolve. Each area has its own color scale, as the units of the two countries differ a lot in size.

In [ ]:
units = {}
for area, regions in AREAS.items():
    config = ADM2[area]
    where = " OR ".join(f"({REGIONS[region]['where']})" for region in regions)
    fields = [config["id"], config["name"], config["province"]]
    gdf = gpd.read_file(gdal_path(config["url"]), where=where, columns=fields)
    gdf = gdf.dissolve(by=fields).reset_index().to_crs("EPSG:4326")
    gdf = gdf.rename(columns={config["id"]: "id", config["name"]: "name", config["province"]: "province"})
    gdf["name"] = gdf["name"].map(display_name)
    gdf["province"] = gdf["province"].map(display_name)

    alerts = firms[area][["instrument", "geometry"]].sjoin(gdf[["id", "geometry"]], predicate="within")
    counts = alerts.groupby(["id", "instrument"]).size().unstack(fill_value=0)
    gdf = gdf.merge(counts, left_on="id", right_index=True, how="left")
    gdf[["VIIRS", "MODIS"]] = gdf[["VIIRS", "MODIS"]].fillna(0).astype(int)
    gdf["alerts"] = gdf["VIIRS"] + gdf["MODIS"]
    gdf["area_km2"] = gdf.to_crs("EPSG:6933").area / 1e6
    gdf["alerts_per_100km2"] = gdf["alerts"] / gdf["area_km2"] * 100
    units[area] = gdf

    # Alerts near the coast or on borders can fall outside every admin level 2 polygon
    print(f"{area}: {len(gdf)} units, {gdf['alerts'].sum():,} of {len(firms[area]):,} alerts assigned")

pd.concat(
    {
        area: gdf.sort_values("alerts", ascending=False)[
            ["name", "province", "VIIRS", "MODIS", "alerts", "area_km2", "alerts_per_100km2"]
        ]
        .head(10)
        .round(1)
        .reset_index(drop=True)
        for area, gdf in units.items()
    }
)

In [ ]:
METRICS = {
    "alerts": "Fire alerts",
    "alerts_per_100km2": "Fire alerts per 100 km²",
}

# One figure per metric, one map per area with its own color scale
for column, label in METRICS.items():
    fig, axes, caxes = area_figure(colorbar="each")
    for area, (ax,) in axes.items():
        unit = ADM2[area]["unit"]
        units[area].plot(
            ax=ax,
            column=column,
            cmap="YlOrRd",
            edgecolor="grey",
            linewidth=0.3,
            legend=True,
            cax=caxes[area][0],
            legend_kwds=dict(label=label),
            zorder=2,
        )
        provinces[area].boundary.plot(ax=ax, color="black", linewidth=1.2, zorder=3)
        add_cities(ax, area)

        set_extent(ax, provinces[area])
        add_basemap(ax, zorder=1)
        decorate(ax, period_title(f"{area} {label.lower()} per {unit}"))
        # Only the city marker, the unit colors are explained by the color bar
        ax.legend(handles=[CITY_HANDLE], **LEGEND)

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / f"{column}_per_adm2_{DATE_NAME}.png", dpi=150, bbox_inches="tight")
    plt.show()